# Parte 2: del LLM al clasificador

**Objetivo:** comparar tres formas de clasificar titulares en politica, economia,
deportes y ciencia_tecnologia:
1. Zero-shot sin entrenar (el piso que hay que superar).
2. Un LLM (Gemini) que etiqueta todo el dataset: supervisión débil.
3. DistilBERT multilingüe afinado con las etiquetas del LLM.

**Referencia para evaluar:** la columna `etiqueta` (sección del diario, Parte 1).
El modelo se entrena con `etiqueta_llm` y se evalúa contra `etiqueta` en un test que no se usa hasta el final.


In [1]:
import json
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import pipeline

c:\GitHub\Python_intermedio_2026\tarea_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
SEMILLA = 42
CARPETA_DATOS = Path("datos")
CARPETA_MODELO = Path("modelo")
CARPETA_MODELO.mkdir(exist_ok=True)

noticias = pd.read_csv(CARPETA_DATOS / "noticias.csv")
noticias["bajada"] = noticias["bajada"].fillna("")
print(noticias.shape)
noticias["etiqueta"].value_counts()

(384, 7)


etiqueta
economia              99
deportes              98
ciencia_tecnologia    95
politica              92
Name: count, dtype: int64

## Paso 1 · Línea base zero-shot
Clasifico 40 titulares (10 por categoría) con `Recognai/bert-base-spanish-wwm-cased-xnli`, un BERT en español entrenado para inferencia (XNLI). No ve ningún ejemplo de nuestras categorías: su F1 es el piso que el modelo afinado debe superar.

In [3]:
muestra_zero_shot = noticias.groupby("etiqueta").sample(10, random_state=SEMILLA)
muestra_zero_shot["etiqueta"].value_counts()

etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64

In [4]:
MODELO_ZERO_SHOT = "Recognai/bert-base-spanish-wwm-cased-xnli"
clasificador_zero_shot = pipeline("zero-shot-classification", model=MODELO_ZERO_SHOT)

c:\GitHub\Python_intermedio_2026\tarea_1\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nofer\.cache\huggingface\hub\models--Recognai--bert-base-spanish-wwm-cased-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\GitHub\Python_intermedio_2026\tarea_1\.venv\Lib\site-packages\huggingface_hub

In [5]:
# El modelo entiende mejor palabras naturales
NOMBRES_NATURALES = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia y tecnología": "ciencia_tecnologia",
}

inicio = time.time()
resultados = clasificador_zero_shot(
    muestra_zero_shot["titular"].tolist(),
    candidate_labels=list(NOMBRES_NATURALES.keys()),
    hypothesis_template="Esta noticia trata sobre {}.",
)
segundos_zero_shot = (time.time() - inicio) / len(muestra_zero_shot)

predicciones = []
for resultado in resultados:
    mejor_opcion = resultado["labels"][0]
    predicciones.append(NOMBRES_NATURALES[mejor_opcion])
muestra_zero_shot["prediccion_zero_shot"] = predicciones

In [7]:
reales = muestra_zero_shot["etiqueta"]
predichas = muestra_zero_shot["prediccion_zero_shot"]
accuracy_zero_shot = accuracy_score(reales, predichas)
f1_zero_shot = f1_score(reales, predichas, average="macro")

print(f"Zero-shot · accuracy: {accuracy_zero_shot:.2f} · F1 macro: {f1_zero_shot:.2f}")
print(f"Tiempo por titular: {segundos_zero_shot:.2f} s")

Zero-shot · accuracy: 0.62 · F1 macro: 0.60
Tiempo por titular: 0.41 s


**Resultado:** sin ningún entrenamiento, el zero-shot acierta el 62 % de los 40 titulares
(F1 macro = 0.60) y tarda 0.41 s por titular. Este es el piso que el modelo afinado debe superar.

## Paso 2 · Supervisión débil con un LLM
Gemini etiqueta todas las noticias con un prompt que define cada categoría. Envío lotes de 20 para no pasar el límite de la capa gratuita, y el avance se guarda en `datos/etiquetas_llm.csv` (titular, etiqueta_llm, confianza, modelo_llm, fecha).
Si el archivo ya existe, no se vuelve a llamar a la API. La clave se lee de `.env`, que no se sube a GitHub.

In [8]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # lee GEMINI_API_KEY del archivo .env
cliente = genai.Client()

In [9]:
for modelo_disponible in cliente.models.list():
    if "flash" in modelo_disponible.name:
        print(modelo_disponible.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [19]:
MODELO_LLM = "gemini-3.5-flash"

INSTRUCCIONES = """Eres editor de un diario peruano. Clasifica cada noticia en UNA categoría:
- politica: Congreso, Gobierno, elecciones, partidos, justicia, relaciones exteriores.
- economia: mercados, empresas, empleo, precios, impuestos, minería, presupuesto.
- deportes: fútbol, vóley, atletismo y cualquier competencia deportiva.
- ciencia_tecnologia: ciencia, espacio, ambiente, tecnología, IA, internet, videojuegos.
Responde SOLO con una lista JSON, un objeto por noticia y en el mismo orden:
[{"id": 0, "etiqueta": "politica", "confianza": 0.95}]
"confianza" va de 0 a 1 e indica qué tan seguro estás."""


def etiquetar_lote(lote):
    """Envía varias noticias en una sola llamada. Devuelve [] si falla dos veces."""
    lineas = []
    for posicion, fila in enumerate(lote.itertuples()):
        lineas.append(f"{posicion}. Titular: {fila.titular} | Bajada: {fila.bajada}")
    texto = INSTRUCCIONES + "\n\nNoticias:\n" + "\n".join(lineas)

    for intento in [1, 2]:
        try:
            respuesta = cliente.models.generate_content(
                model=MODELO_LLM,
                contents=texto,
                config=types.GenerateContentConfig(
                    temperature=0, response_mime_type="application/json"
                ),
            )
            return json.loads(respuesta.text)
        except Exception as error:
            print("Falló el intento", intento, ":", error)
            time.sleep(30)
    return []

In [16]:
inicio = time.time()
prueba_llm = etiquetar_lote(noticias.head(3))
segundos_llm = (time.time() - inicio) / 3
print(prueba_llm)
print(f"Tiempo por titular: {segundos_llm:.2f} s")

[{'id': 0, 'etiqueta': 'deportes', 'confianza': 0.99}, {'id': 1, 'etiqueta': 'deportes', 'confianza': 0.99}, {'id': 2, 'etiqueta': 'deportes', 'confianza': 0.98}]
Tiempo por titular: 3.46 s


In [20]:
archivo_llm = CARPETA_DATOS / "etiquetas_llm.csv"
if archivo_llm.exists():
    etiquetas_llm = pd.read_csv(archivo_llm)
else:
    columnas_llm = ["titular", "etiqueta_llm", "confianza", "modelo_llm", "fecha"]
    etiquetas_llm = pd.DataFrame(columns=columnas_llm)

pendientes = noticias[~noticias["titular"].isin(etiquetas_llm["titular"])]
print("Noticias por etiquetar:", len(pendientes))

Noticias por etiquetar: 84


In [21]:
TAMANO_LOTE = 20
for inicio_lote in range(0, len(pendientes), TAMANO_LOTE):
    lote = pendientes.iloc[inicio_lote:inicio_lote + TAMANO_LOTE]
    nuevas = []
    for respuesta in etiquetar_lote(lote):
        nuevas.append({
            "titular": lote.iloc[int(respuesta["id"])]["titular"],
            "etiqueta_llm": respuesta["etiqueta"],
            "confianza": respuesta["confianza"],
            "modelo_llm": MODELO_LLM,
            "fecha": date.today().isoformat(),
        })
    etiquetas_llm = pd.concat([etiquetas_llm, pd.DataFrame(nuevas)], ignore_index=True)
    etiquetas_llm.to_csv(archivo_llm, index=False, encoding="utf-8")
    print("Etiquetadas:", len(etiquetas_llm), "de", len(noticias))
    time.sleep(6)  # respeta el límite de solicitudes por minuto de la capa gratuita

Etiquetadas: 320 de 384
Falló el intento 1 : 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Etiquetadas: 340 de 384
Falló el intento 1 : 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Etiquetadas: 360 de 384
Etiquetadas: 380 de 384
Etiquetadas: 384 de 384


**Nota sobre el límite gratuito de Gemini:** la capa gratuita permite 20 solicitudes por día
para cada modelo. Las primeras 300 noticias se etiquetaron con `gemini-3.8-flash` (15 lotes) y, al agotarse su cuota, las 84 restantes con `gemini-3.5-flash`. La columna `modelo_llm` de `etiquetas_llm.csv` registra qué modelo etiquetó cada noticia. Durante el proceso también hubo errores temporales 503 (alta demanda), que se resolvieron con el reintento automático.

In [22]:
fuera_de_lista = etiquetas_llm[~etiquetas_llm["etiqueta_llm"].isin(CATEGORIAS)]
print("Etiquetas fuera de las 4 categorías:", len(fuera_de_lista))
print("Noticias por modelo:", etiquetas_llm["modelo_llm"].value_counts().to_dict())

# Si ejecutas esta celda dos veces, primero se quitan las columnas de la vez anterior
noticias = noticias.drop(columns=["etiqueta_llm", "confianza"], errors="ignore")
columnas_unir = ["titular", "etiqueta_llm", "confianza"]
noticias = noticias.merge(etiquetas_llm[columnas_unir], on="titular", how="left")
coincide_con_seccion = (noticias["etiqueta_llm"] == noticias["etiqueta"]).mean()
print(f"El LLM coincide con la sección del diario en el {coincide_con_seccion:.0%} de noticias")
noticias["etiqueta_llm"].value_counts()

Etiquetas fuera de las 4 categorías: 0
Noticias por modelo: {'gemini-3.8-flash': 300, 'gemini-3.5-flash': 84}
El LLM coincide con la sección del diario en el 92% de noticias


etiqueta_llm
politica              103
deportes               98
economia               95
ciencia_tecnologia     88
Name: count, dtype: int64

**Resultado:** Gemini etiquetó las 384 noticias, todas en una de las 4 categorías. Sus etiquetas coinciden con la sección del diario en el 92% de los casos.

### Auditoría humana de 20 casos
Leo 20 noticias elegidas al azar (`random_state=42`) y escribo mi categoría sin mirar la del LLM. Luego cuento en cuántas coincidimos y comento los desacuerdos.

In [23]:
pd.set_option("display.max_colwidth", 150)
auditoria = noticias.sample(20, random_state=SEMILLA)[["titular", "etiqueta_llm"]]
auditoria = auditoria.reset_index(drop=True)
auditoria

,titular,etiqueta_llm
0,Carlos Mesía: “En dos semanas debe estar debatiéndose [el dictamen de facultades] en el pleno del Senado”,politica
1,¿Lima puede elegir a su alcalde en segunda vuelta? Esto tendría que cambiar para aplicar en el futuro propuesta de Francis Allison,politica
2,NASA: los cuatro astronautas de Crew 12 retornaron a la tierra,ciencia_tecnologia
3,Científicos descubren cómo predecir cuándo una inteligencia artificial dará respuestas peligrosas,ciencia_tecnologia
4,"“El día de mañana me encantaría ver a tantas niñas entrando al automovilismo”: Annia Cillóniz, líder en la categoría S1600, sobre sus sueños en Ca...",deportes
5,Noruega estudia prohibir temporalmente las gafas inteligentes,ciencia_tecnologia
6,"Neymar se rinde ante Lamine Yamal: ""Es el mejor del mundo""",deportes
7,"El dólar abre a la baja en S/3,42 ante debilitamiento de la divisa estadounidense",economia
8,"Las empresas de IA son incapaces de controlar sus modelos, alerta un exinvestigador de OpenAI",ciencia_tecnologia
9,Keiko Fujimori ratifica su respaldo al ministro César Astudillo y al jefe de la PNP Fredy López,politica


In [24]:
etiquetas_humanas = [
    "politica",  # 0
    "politica",  # 1
    "ciencia_tecnologia",  # 2
    "ciencia_tecnologia",  # 3
    "deportes",  # 4
    "ciencia_tecnologia",  # 5
    "deportes",  # 6
    "economia",  # 7
    "ciencia_tecnologia",  # 8
    "politica",  # 9
    "politica",  # 10
    "economia",  # 11
    "economia",  # 12
    "politica",  # 13
    "economia",  # 14
    "economia",  # 15
    "economia",  # 16
    "ciencia_tecnologia",  # 17
    "ciencia_tecnologia",  # 18
    "politica",  # 19
]


In [25]:
auditoria["etiqueta_humana"] = etiquetas_humanas
auditoria["coinciden"] = auditoria["etiqueta_llm"] == auditoria["etiqueta_humana"]
aciertos = int(auditoria["coinciden"].sum())
print(f"Acuerdo LLM–humano: {aciertos}/20 = {aciertos / 20:.0%}")
auditoria[~auditoria["coinciden"]]

Acuerdo LLM–humano: 20/20 = 100%


,titular,etiqueta_llm,etiqueta_humana,coinciden


**Resultado de la auditoría:** coincidí con el LLM en 20 de 20 casos (100 %). Aun así, el LLM no coincide con la sección del diario en 30 de las 384 noticias (8 %). En deportes no hay ningún desacuerdo; la mayoría son temas que cruzan categorías, como la Ley MAPE de minería (sección política, el LLM dice economía) o los sistemas de alerta de El Niño (sección tecnología, el LLM dice política). La sección del diario no siempre describe el tema del titular.